In [ ]:
from pathlib import Path
import json
import random
from types import SimpleNamespace

import torch

from src.utils.config_loader import load_task_configs, print_config
from src.utils.config_loader import load_config

BASE_CONFIG = load_config("configs/config.yaml")
OBJECT_CONFIG = load_task_configs("configs/config.yaml", "configs/object_config.yaml")
ATTRIBUTE_CONFIG = load_task_configs("configs/config.yaml", "configs/attribute_config.yaml")
TASK2_CONFIG = load_task_configs("configs/config.yaml", "configs/relation_config.yaml")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / BASE_CONFIG.paths.data_dir
RAW_DIR = PROJECT_ROOT / BASE_CONFIG.paths.raw_dir


def normalize_pipeline_mode(mode):
    mode = str(mode).lower().strip()
    aliases = {
        "both": "all",
        "task2": "relation",
    }
    return aliases.get(mode, mode)


def normalize_input_mode(mode):
    mode = str(mode).lower().strip()
    aliases = {
        "grayscale": "gray",
        "grey": "gray",
        "edge": "contour",
        "edges": "contour",
    }
    mode = aliases.get(mode, mode)
    if mode not in {"rgb", "gray", "contour"}:
        raise ValueError("input mode pháº£i lÃ  'rgb', 'gray', hoáº·c 'contour'")
    return mode


def ns(**kwargs):
    return SimpleNamespace(**kwargs)


TASK1_CONFIG = ns(
    backbone=ns(
        name=str(OBJECT_CONFIG.backbone.name),
        pretrained=bool(OBJECT_CONFIG.backbone.pretrained),
        learnable_backbone=bool(OBJECT_CONFIG.backbone.learnable_backbone),
        freeze_backbone=bool(OBJECT_CONFIG.backbone.freeze_backbone),
        freeze_epochs=int(OBJECT_CONFIG.backbone.freeze_epochs),
        feature_dim=int(OBJECT_CONFIG.backbone.feature_dim),
    ),
    dataset=ns(
        processed_dir="data/processed/task1",
        feature_cache_dir=str(OBJECT_CONFIG.dataset.feature_cache_dir),
        use_feature_cache=bool(OBJECT_CONFIG.dataset.use_feature_cache),
        object_input_mode=str(OBJECT_CONFIG.dataset.object_input_mode),
        attribute_input_mode=str(ATTRIBUTE_CONFIG.dataset.attribute_input_mode),
    ),
    labels=ns(
        max_objects=int(OBJECT_CONFIG.labels.max_objects),
        max_attributes=int(ATTRIBUTE_CONFIG.labels.max_attributes),
    ),
    model=ns(
        object_hidden_dim=int(OBJECT_CONFIG.model.object_hidden_dim),
        object_dropout=float(OBJECT_CONFIG.model.object_dropout),
        object_num_layers=int(OBJECT_CONFIG.model.object_num_layers),
        attribute_hidden_dim=int(ATTRIBUTE_CONFIG.model.attribute_hidden_dim),
        attribute_dropout=float(ATTRIBUTE_CONFIG.model.attribute_dropout),
        attribute_num_layers=int(ATTRIBUTE_CONFIG.model.attribute_num_layers),
    ),
    loss=ns(
        object_weight=float(OBJECT_CONFIG.loss.object_weight),
        attribute_weight=float(ATTRIBUTE_CONFIG.loss.attribute_weight),
        attribute_pos_weight=float(ATTRIBUTE_CONFIG.loss.attribute_pos_weight),
    ),
    training=ns(
        batch_size=int(OBJECT_CONFIG.training.batch_size),
        lr=float(OBJECT_CONFIG.training.lr),
        max_epochs=int(OBJECT_CONFIG.training.max_epochs),
    ),
    augmentation=ns(
        random_horizontal_flip=float(OBJECT_CONFIG.augmentation.random_horizontal_flip),
        color_jitter=ns(
            enabled=bool(OBJECT_CONFIG.augmentation.color_jitter.enabled),
            brightness=float(OBJECT_CONFIG.augmentation.color_jitter.brightness),
            contrast=float(OBJECT_CONFIG.augmentation.color_jitter.contrast),
            saturation=float(OBJECT_CONFIG.augmentation.color_jitter.saturation),
            hue=float(OBJECT_CONFIG.augmentation.color_jitter.hue),
        ),
        random_erasing_p=float(OBJECT_CONFIG.augmentation.random_erasing_p),
        resize_delta=int(OBJECT_CONFIG.augmentation.resize_delta),
    ),
    eval=ns(
        object_top_k=list(OBJECT_CONFIG.eval.object_top_k),
        attribute_threshold_mode=str(ATTRIBUTE_CONFIG.eval.attribute_threshold_mode),
        attribute_threshold=float(ATTRIBUTE_CONFIG.eval.attribute_threshold),
        attribute_threshold_scale=float(ATTRIBUTE_CONFIG.eval.attribute_threshold_scale),
        attribute_threshold_min=float(ATTRIBUTE_CONFIG.eval.attribute_threshold_min),
        attribute_threshold_max=float(ATTRIBUTE_CONFIG.eval.attribute_threshold_max),
    ),
)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / BASE_CONFIG.paths.data_dir
RAW_DIR = PROJECT_ROOT / BASE_CONFIG.paths.raw_dir

TRAINING_MODE = normalize_pipeline_mode(BASE_CONFIG.pipeline.training_mode)
DOWNLOAD_DATA = bool(BASE_CONFIG.pipeline.download_data)
STRICT_SAMPLE_MODE = bool(BASE_CONFIG.sampling.strict_mode)
SAMPLE_SIZE = int(BASE_CONFIG.sampling.sample_size)
SAMPLE_SPLIT_RATIOS = tuple(float(x) for x in BASE_CONFIG.sampling.split_ratios)
SAMPLE_SEED = int(BASE_CONFIG.sampling.seed)
IMAGE_DOWNLOAD_MODE = "none" if STRICT_SAMPLE_MODE else str(BASE_CONFIG.sampling.image_download_mode)
PRE_EXTRACT_FEATURES = bool(BASE_CONFIG.pipeline.pre_extract_features)

FEATURE_BATCH_SIZE = int(BASE_CONFIG.feature_extraction.batch_size)
FEATURE_RESIZE_SIZE = int(BASE_CONFIG.feature_extraction.resize_size)
FEATURE_CROP_SIZE = int(BASE_CONFIG.feature_extraction.crop_size)
FEATURE_MEAN = [float(x) for x in BASE_CONFIG.image.mean]
FEATURE_STD = [float(x) for x in BASE_CONFIG.image.std]
FULL_SPLIT_RATIOS = (float(BASE_CONFIG.split.train), float(BASE_CONFIG.split.val), float(BASE_CONFIG.split.test))
PREPROCESS_SPLIT_RATIOS = SAMPLE_SPLIT_RATIOS if STRICT_SAMPLE_MODE else FULL_SPLIT_RATIOS

MAX_SAMPLES = None if BASE_CONFIG.sampling.max_samples is None else int(BASE_CONFIG.sampling.max_samples)

if STRICT_SAMPLE_MODE:
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_sample"
else:
    PROCESSED_DIR = PROJECT_ROOT / BASE_CONFIG.paths.processed_dir

CHECKPOINT_DIR = PROJECT_ROOT / BASE_CONFIG.paths.checkpoint_dir
LOG_DIR = PROJECT_ROOT / BASE_CONFIG.paths.log_dir

TASK1_PROCESSED_DIR = PROCESSED_DIR / "task1"
TASK2_PROCESSED_DIR = PROCESSED_DIR / "task2"
TASK1_FEATURE_DIR = TASK1_PROCESSED_DIR / TASK1_CONFIG.dataset.feature_cache_dir
TASK2_FEATURE_DIR = TASK2_PROCESSED_DIR / TASK2_CONFIG.dataset.feature_cache_dir

LEGACY_TASK1_MODE = str(BASE_CONFIG.pipeline.training_mode).lower().strip() == "task1"
RUN_OBJECT_PIPELINE = TRAINING_MODE in {"object", "all", "e2e"} or LEGACY_TASK1_MODE
RUN_ATTRIBUTE_PIPELINE = TRAINING_MODE in {"attribute", "all", "e2e"} or LEGACY_TASK1_MODE
RUN_RELATION_PIPELINE = TRAINING_MODE in {"relation", "all", "e2e"}
RUN_E2E_WRAPPER = TRAINING_MODE in {"e2e", "all"}

TASK1_LEARNABLE_BACKBONE = bool(TASK1_CONFIG.backbone.learnable_backbone)
TASK2_LEARNABLE_BACKBONE = bool(TASK2_CONFIG.backbone.learnable_backbone)

TASK1_OBJECT_INPUT_MODE = normalize_input_mode(TASK1_CONFIG.dataset.object_input_mode)
TASK1_ATTRIBUTE_INPUT_MODE = normalize_input_mode(TASK1_CONFIG.dataset.attribute_input_mode)
TASK2_INPUT_MODE = normalize_input_mode(TASK2_CONFIG.dataset.input_mode)

TASK1_OBJECT_USE_CACHE = bool(TASK1_CONFIG.dataset.use_feature_cache) and not TASK1_LEARNABLE_BACKBONE
TASK1_ATTRIBUTE_USE_CACHE = bool(TASK1_CONFIG.dataset.use_feature_cache) and not TASK1_LEARNABLE_BACKBONE
TASK2_USE_CACHE = bool(TASK2_CONFIG.dataset.use_feature_cache) and not TASK2_LEARNABLE_BACKBONE

TASK1_BATCH_SIZE = int(TASK1_CONFIG.training.batch_size)
TASK2_BATCH_SIZE = int(TASK2_CONFIG.training.batch_size)
TASK1_LR = float(TASK1_CONFIG.training.lr)
TASK2_LR = float(TASK2_CONFIG.training.lr)
TASK1_MAX_EPOCHS = int(TASK1_CONFIG.training.max_epochs)
TASK2_MAX_EPOCHS = int(TASK2_CONFIG.training.max_epochs)
TASK1_ROI_SIZE = int(BASE_CONFIG.image.roi_size)
TASK2_ROI_SIZE = int(BASE_CONFIG.image.roi_size)

TASK1_OBJECT_HIDDEN_DIM = int(TASK1_CONFIG.model.object_hidden_dim)
TASK1_OBJECT_DROPOUT = float(TASK1_CONFIG.model.object_dropout)
TASK1_OBJECT_NUM_LAYERS = int(TASK1_CONFIG.model.object_num_layers)
TASK1_ATTRIBUTE_HIDDEN_DIM = int(TASK1_CONFIG.model.attribute_hidden_dim)
TASK1_ATTRIBUTE_DROPOUT = float(TASK1_CONFIG.model.attribute_dropout)
TASK1_ATTRIBUTE_NUM_LAYERS = int(TASK1_CONFIG.model.attribute_num_layers)
TASK1_OBJECT_WEIGHT = float(TASK1_CONFIG.loss.object_weight)
TASK1_ATTRIBUTE_WEIGHT = float(TASK1_CONFIG.loss.attribute_weight)
TASK1_ATTRIBUTE_POS_WEIGHT = float(TASK1_CONFIG.loss.attribute_pos_weight)

TASK2_FUSION_METHOD = str(TASK2_CONFIG.model.fusion_method)
TASK2_HIDDEN_DIM = int(TASK2_CONFIG.model.hidden_dim)
TASK2_DROPOUT = float(TASK2_CONFIG.model.dropout)
TASK2_NUM_LAYERS = int(TASK2_CONFIG.model.num_layers)
TASK2_ATTENTION_HEADS = int(TASK2_CONFIG.model.attention_heads)

DEVICE = BASE_CONFIG.device if BASE_CONFIG.device == "cpu" or torch.cuda.is_available() else "cpu"
print_config(BASE_CONFIG)
print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {DEVICE}")
print(f"Pipeline mode: {TRAINING_MODE}")
print(f"Strict sample mode: {STRICT_SAMPLE_MODE}")
print(f"Sample size: {SAMPLE_SIZE} | split ratios: {SAMPLE_SPLIT_RATIOS} | seed: {SAMPLE_SEED}")
print(f"Processed root: {PROCESSED_DIR}")
print(f"Feature extraction batch size: {FEATURE_BATCH_SIZE}")
print(f"Task 1 learnable backbone: {TASK1_LEARNABLE_BACKBONE}")
print(f"Task 2 learnable backbone: {TASK2_LEARNABLE_BACKBONE}")
print(f"Task 1 object input mode: {TASK1_OBJECT_INPUT_MODE}")
print(f"Task 1 attribute input mode: {TASK1_ATTRIBUTE_INPUT_MODE}")
print(f"Task 2 input mode: {TASK2_INPUT_MODE}")
print(f"Task 1 config: batch_size={TASK1_BATCH_SIZE}, lr={TASK1_LR}, max_epochs={TASK1_MAX_EPOCHS}")
print(f"Task 2 config: batch_size={TASK2_BATCH_SIZE}, lr={TASK2_LR}, max_epochs={TASK2_MAX_EPOCHS}")


def require_files(paths, label):
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f"Thiáº¿u {label}: {missing}")


def ensure_nonempty_cache(cache_path):
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return False
    if cache_path.stat().st_size == 0:
        return False
    try:
        cache = torch.load(cache_path, map_location="cpu")
        return bool(cache)
    except Exception:
        return False


def get_feature_output(task_name, split_name, suffix=None):
    base_dir = TASK1_FEATURE_DIR if task_name == "task1" else TASK2_FEATURE_DIR
    filename = f"{split_name}_features.pt" if suffix is None else f"{split_name}_{suffix}_features.pt"
    return base_dir / filename

In [ ]:
# Cáº¥u hÃ¬nh cháº¡y náº±m trong configs/config.yaml, configs/object_config.yaml, configs/attribute_config.yaml và configs/relation_config.yaml.
# Cell nÃ y chá»‰ lÃ  ghi chÃº; cÃ¡c biáº¿n thá»±c thi sáº½ Ä‘Æ°á»£c náº¡p tá»« config á»Ÿ cell sau.

# Chá»‰nh config YAML thay vÃ¬ sá»­a giÃ¡ trá»‹ trá»±c tiáº¿p trong notebook.

Cáº¥u hÃ¬nh cháº¡y náº±m trong `configs/config.yaml`, `configs/object_config.yaml`, `configs/attribute_config.yaml` vÃ  `configs/relation_config.yaml`.

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
%pip install transformers timm datasets tqdm omegaconf scikit-learn pyyaml wandb
%pip install opencv-python pillow matplotlib seaborn

In [ ]:
import os
import sys
import torch

current_dir = os.path.abspath(os.getcwd())
if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

to_remove = [k for k in sys.modules.keys() if k == 'src' or k.startswith('src.')]
for k in to_remove:
    del sys.modules[k]

print(f"Project Root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {DEVICE}")

## 2. Load Data

In [ ]:
from src.data.download import download_and_extract_metadata, download_vg_images, verify_download

image_dir = PROJECT_ROOT / BASE_CONFIG.dataset.image_dir

if DOWNLOAD_DATA:
    print("Äang táº£i metadata Visual Genome...")
    download_and_extract_metadata(raw_dir=str(RAW_DIR), keep_zip=False)

    raw_status = verify_download(raw_dir=str(RAW_DIR))
    missing_metadata = [name for name, ok in raw_status.items() if not ok]
    if missing_metadata:
        raise RuntimeError(f"Thiáº¿u metadata sau khi táº£i: {missing_metadata}")

    with open(RAW_DIR / "image_data.json", "r", encoding="utf-8") as f:
        img_data = json.load(f)

    if IMAGE_DOWNLOAD_MODE == 'sample':
        sample_ids = [img['image_id'] for img in img_data[:SAMPLE_SIZE]]
        print(f"Báº¯t Ä‘áº§u táº£i bá»™ sample {len(sample_ids)} áº£nh...")
        downloaded_images = download_vg_images(sample_ids, image_dir=str(image_dir))
        if not downloaded_images:
            raise RuntimeError("KhÃ´ng táº£i Ä‘Æ°á»£c áº£nh máº«u nÃ o.")
    elif IMAGE_DOWNLOAD_MODE == 'all':
        all_ids = [img['image_id'] for img in img_data]
        print(f"Báº¯t Ä‘áº§u táº£i toÃ n bá»™ {len(all_ids)} áº£nh...")
        downloaded_images = download_vg_images(all_ids, image_dir=str(image_dir))
        if not downloaded_images:
            raise RuntimeError("KhÃ´ng táº£i Ä‘Æ°á»£c áº£nh nÃ o.")
    elif IMAGE_DOWNLOAD_MODE == 'none':
        print("Bá» qua táº£i áº£nh theo cáº¥u hÃ¬nh.")
    else:
        raise ValueError("IMAGE_DOWNLOAD_MODE pháº£i lÃ  'none', 'sample', hoáº·c 'all'.")
else:
    print("Bá» qua táº£i má»›i, chá»‰ kiá»ƒm tra dá»¯ liá»‡u hiá»‡n cÃ³...")
    raw_status = verify_download(raw_dir=str(RAW_DIR))
    missing_metadata = [name for name, ok in raw_status.items() if not ok]
    if missing_metadata:
        raise RuntimeError(
            "Thiáº¿u dá»¯ liá»‡u RAW cáº§n thiáº¿t: " + ", ".join(missing_metadata) +
            ". HÃ£y báº­t DOWNLOAD_DATA = True hoáº·c Ä‘áº·t Ä‘Ãºng thÆ° má»¥c data/raw."
        )

In [ ]:
if STRICT_SAMPLE_MODE:
    image_data_file = RAW_DIR / "image_data.json"
    require_files([image_data_file], "image_data.json")

    with open(image_data_file, "r", encoding="utf-8") as f:
        image_data = json.load(f)

    all_image_ids = [img["image_id"] for img in image_data]
    sample_count = min(SAMPLE_SIZE, len(all_image_ids))
    SAMPLE_IMAGE_IDS = random.Random(SAMPLE_SEED).sample(all_image_ids, sample_count)
    print(f"ÄÃ£ chá»n trÆ°á»›c {len(SAMPLE_IMAGE_IDS)} image_id cho sample strict.")
else:
    SAMPLE_IMAGE_IDS = None
    print("Bá» qua strict sample; sáº½ dÃ¹ng toÃ n bá»™ dá»¯ liá»‡u theo split máº·c Ä‘á»‹nh.")

## 3. Data Preprocessing (Táº¡o Vocab & Split)

In [ ]:
from src.data.preprocessing import build_vocab_and_splits

if RUN_OBJECT_PIPELINE or RUN_ATTRIBUTE_PIPELINE or RUN_E2E_WRAPPER:
    build_vocab_and_splits(
        task='task1',
        raw_dir=str(RAW_DIR),
        processed_dir=str(PROCESSED_DIR),
        max_objects=int(TASK1_CONFIG.labels.max_objects),
        max_attributes=int(TASK1_CONFIG.labels.max_attributes),
        sample_image_ids=SAMPLE_IMAGE_IDS if STRICT_SAMPLE_MODE else None,
        split_by_image_id=STRICT_SAMPLE_MODE,
        split_ratios=PREPROCESS_SPLIT_RATIOS,
        seed=SAMPLE_SEED,
    )
    require_files(
        [
            TASK1_PROCESSED_DIR / 'object_vocab.json',
            TASK1_PROCESSED_DIR / 'attribute_vocab.json',
            TASK1_PROCESSED_DIR / 'train' / 'annotations.json',
            TASK1_PROCESSED_DIR / 'val' / 'annotations.json',
            TASK1_PROCESSED_DIR / 'test' / 'annotations.json',
        ],
        'Task 1 processed files'
    )

if RUN_RELATION_PIPELINE or RUN_E2E_WRAPPER:
    build_vocab_and_splits(
        task='task2',
        raw_dir=str(RAW_DIR),
        processed_dir=str(PROCESSED_DIR),
        max_relations=int(TASK2_CONFIG.labels.max_relations),
        sample_image_ids=SAMPLE_IMAGE_IDS if STRICT_SAMPLE_MODE else None,
        split_by_image_id=STRICT_SAMPLE_MODE,
        split_ratios=PREPROCESS_SPLIT_RATIOS,
        seed=SAMPLE_SEED,
    )
    require_files(
        [
            TASK2_PROCESSED_DIR / 'relation_vocab.json',
            TASK2_PROCESSED_DIR / 'train' / 'annotations.json',
            TASK2_PROCESSED_DIR / 'val' / 'annotations.json',
            TASK2_PROCESSED_DIR / 'test' / 'annotations.json',
        ],
        'Task 2 processed files'
    )

## 4. TrÃ­ch Xuáº¥t Äáº·c TrÆ°ng (Feature Extraction)
Chá»‰ xuáº¥t Ä‘áº·c trÆ°ng tá»« nhá»¯ng áº£nh `.jpg` báº¡n Ä‘Ã£ táº£i vá» thÆ° má»¥c `images`.

In [ ]:
from pathlib import Path
import json

from src.data.download import download_vg_images
from src.features.feature_extractor import extract_task1_features, extract_task2_features


def collect_split_image_ids(processed_dir, split_names):
    image_ids = set()
    for split_name in split_names:
        annotation_file = Path(processed_dir) / split_name / 'annotations.json'
        if not annotation_file.exists():
            raise FileNotFoundError(f"Thiáº¿u annotation: {annotation_file}")

        with open(annotation_file, 'r', encoding='utf-8') as f:
            raw = json.load(f)

        samples = raw if isinstance(raw, list) else raw.get('samples', [])
        image_ids.update(sample['image_id'] for sample in samples)

    return sorted(image_ids)


def ensure_images_for_splits(task_name, processed_dir, split_names):
    image_ids = collect_split_image_ids(processed_dir, split_names)
    missing_ids = [img_id for img_id in image_ids if not (image_dir / f"{img_id}.jpg").exists()]

    print(f"[{task_name}] Tá»•ng áº£nh tham chiáº¿u: {len(image_ids)} | thiáº¿u local: {len(missing_ids)}")
    if missing_ids:
        print(f"[{task_name}] Äang táº£i bá»• sung áº£nh cÃ²n thiáº¿u...")
        downloaded = download_vg_images(missing_ids, image_dir=str(image_dir))
        if len(downloaded) < len(missing_ids):
            print(f"[Warning] {task_name}: cÃ²n {len(missing_ids) - len(downloaded)} áº£nh chÆ°a táº£i Ä‘Æ°á»£c.")


def cache_output(task_name, split_name, input_mode):
    suffix = None if input_mode == 'rgb' else input_mode
    return get_feature_output(task_name, split_name, suffix=suffix)


if PRE_EXTRACT_FEATURES:
    TASK1_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
    TASK2_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

    if (RUN_OBJECT_PIPELINE or RUN_ATTRIBUTE_PIPELINE or RUN_E2E_WRAPPER) and (TASK1_OBJECT_USE_CACHE or TASK1_ATTRIBUTE_USE_CACHE):
        ensure_images_for_splits('Task 1', TASK1_PROCESSED_DIR, ['train', 'val', 'test'])
        print('Extracting features Task 1...')
        for split_name in ['train', 'val', 'test']:
            if TASK1_OBJECT_USE_CACHE:
                object_output = cache_output('task1', split_name, TASK1_OBJECT_INPUT_MODE)
                extract_task1_features(
                    annotation_file=str(TASK1_PROCESSED_DIR / split_name / 'annotations.json'),
                    image_dir=str(image_dir),
                    output_file=str(object_output),
                    backbone=str(TASK1_CONFIG.backbone.name),
                    pretrained=bool(TASK1_CONFIG.backbone.pretrained),
                    batch_size=FEATURE_BATCH_SIZE,
                    device=DEVICE,
                    resize_size=FEATURE_RESIZE_SIZE,
                    crop_size=FEATURE_CROP_SIZE,
                    mean=FEATURE_MEAN,
                    std=FEATURE_STD,
                    input_mode=TASK1_OBJECT_INPUT_MODE,
                )
                if not ensure_nonempty_cache(object_output):
                    raise RuntimeError(f'Task 1 object feature cache rá»—ng: {object_output}')

            if TASK1_ATTRIBUTE_USE_CACHE and TASK1_ATTRIBUTE_INPUT_MODE != TASK1_OBJECT_INPUT_MODE:
                attribute_output = cache_output('task1', split_name, TASK1_ATTRIBUTE_INPUT_MODE)
                extract_task1_features(
                    annotation_file=str(TASK1_PROCESSED_DIR / split_name / 'annotations.json'),
                    image_dir=str(image_dir),
                    output_file=str(attribute_output),
                    backbone=str(TASK1_CONFIG.backbone.name),
                    pretrained=bool(TASK1_CONFIG.backbone.pretrained),
                    batch_size=FEATURE_BATCH_SIZE,
                    device=DEVICE,
                    resize_size=FEATURE_RESIZE_SIZE,
                    crop_size=FEATURE_CROP_SIZE,
                    mean=FEATURE_MEAN,
                    std=FEATURE_STD,
                    input_mode=TASK1_ATTRIBUTE_INPUT_MODE,
                )
                if not ensure_nonempty_cache(attribute_output):
                    raise RuntimeError(f'Task 1 attribute feature cache rá»—ng: {attribute_output}')

    if RUN_RELATION_PIPELINE or RUN_E2E_WRAPPER:
        if TASK2_USE_CACHE:
            ensure_images_for_splits('Task 2', TASK2_PROCESSED_DIR, ['train', 'val', 'test'])
            print('Extracting features Task 2...')
            for split_name in ['train', 'val', 'test']:
                relation_output = cache_output('task2', split_name, TASK2_INPUT_MODE)
                extract_task2_features(
                    annotation_file=str(TASK2_PROCESSED_DIR / split_name / 'annotations.json'),
                    image_dir=str(image_dir),
                    output_file=str(relation_output),
                    backbone=str(TASK2_CONFIG.backbone.name),
                    pretrained=bool(TASK2_CONFIG.backbone.pretrained),
                    batch_size=FEATURE_BATCH_SIZE,
                    device=DEVICE,
                    resize_size=FEATURE_RESIZE_SIZE,
                    crop_size=FEATURE_CROP_SIZE,
                    mean=FEATURE_MEAN,
                    std=FEATURE_STD,
                    input_mode=TASK2_INPUT_MODE,
                )
                if not ensure_nonempty_cache(relation_output):
                    raise RuntimeError(f'Task 2 feature cache rá»—ng: {relation_output}')
else:
    print('Bá» qua pre-extraction theo cáº¥u hÃ¬nh PRE_EXTRACT_FEATURES = False.')

## 5. Pipeline 1: Object Classification

In [ ]:
if RUN_OBJECT_PIPELINE:
    print("========== PIPELINE 1: OBJECT CLASSIFICATION ==========")
    from src.training.object_trainer import ObjectTrainer
    from src.models.task1.object_classifier import ObjectClassifier
    from src.data.task1_dataset import build_task1_datasets
    from torch.utils.data import DataLoader

    if not TASK1_LEARNABLE_BACKBONE and not TASK1_OBJECT_USE_CACHE:
        raise ValueError("Task 1 object pipeline requires feature cache when learnable_backbone=false.")

    print("Khá»Ÿi táº¡o Dataset cho object pipeline...")
    object_train_ds, object_val_ds, object_test_ds = build_task1_datasets(
        processed_dir=str(TASK1_PROCESSED_DIR),
        image_dir=str(image_dir),
        roi_size=TASK1_ROI_SIZE,
        use_feature_cache=TASK1_OBJECT_USE_CACHE,
        feature_cache_dir=str(TASK1_FEATURE_DIR),
        input_mode=TASK1_OBJECT_INPUT_MODE,
        max_samples=MAX_SAMPLES,
        train_horizontal_flip_p=float(TASK1_CONFIG.augmentation.random_horizontal_flip),
        train_color_jitter=bool(TASK1_CONFIG.augmentation.color_jitter.enabled),
        train_brightness=float(TASK1_CONFIG.augmentation.color_jitter.brightness),
        train_contrast=float(TASK1_CONFIG.augmentation.color_jitter.contrast),
        train_saturation=float(TASK1_CONFIG.augmentation.color_jitter.saturation),
        train_hue=float(TASK1_CONFIG.augmentation.color_jitter.hue),
        train_random_erasing_p=float(TASK1_CONFIG.augmentation.random_erasing_p),
        train_resize_delta=int(TASK1_CONFIG.augmentation.resize_delta),
        mean=FEATURE_MEAN,
        std=FEATURE_STD,
    )

    object_model = ObjectClassifier(
        num_classes=object_train_ds.num_objects,
        feature_dim=int(TASK1_CONFIG.backbone.feature_dim),
        hidden_dim=TASK1_OBJECT_HIDDEN_DIM,
        dropout=TASK1_OBJECT_DROPOUT,
        num_layers=TASK1_OBJECT_NUM_LAYERS,
        backbone_name=str(TASK1_CONFIG.backbone.name) if TASK1_LEARNABLE_BACKBONE else None,
        pretrained=bool(TASK1_CONFIG.backbone.pretrained),
        freeze_backbone=bool(TASK1_CONFIG.backbone.freeze_backbone),
        learnable_backbone=TASK1_LEARNABLE_BACKBONE,
        device=DEVICE,
    )

    object_optimizer = torch.optim.AdamW(
        object_model.parameters(),
        lr=TASK1_LR,
        weight_decay=float(BASE_CONFIG.optimizer.weight_decay),
    )

    object_train_loader = DataLoader(
        object_train_ds,
        batch_size=TASK1_BATCH_SIZE,
        shuffle=True,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    object_val_loader = DataLoader(
        object_val_ds,
        batch_size=TASK1_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    object_test_loader = DataLoader(
        object_test_ds,
        batch_size=TASK1_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )

    object_trainer = ObjectTrainer(
        model=object_model,
        train_loader=object_train_loader,
        val_loader=object_val_loader,
        optimizer=object_optimizer,
        use_auto_class_weights=True,
        freeze_backbone=bool(TASK1_CONFIG.backbone.freeze_backbone),
        freeze_epochs=int(TASK1_CONFIG.backbone.freeze_epochs),
        max_epochs=TASK1_MAX_EPOCHS,
        early_stopping_patience=int(BASE_CONFIG.training.early_stopping_patience),
        gradient_clip_val=float(BASE_CONFIG.training.gradient_clip_val),
        log_every_n_steps=int(BASE_CONFIG.training.log_every_n_steps),
        use_amp=(DEVICE == 'cuda'),
    )

    print("Báº¯t Ä‘áº§u training object pipeline...")
    object_val_metrics = object_trainer.train()
    object_best_checkpoint = object_trainer.checkpoint_manager.get_best_checkpoint_path()
    print("Object pipeline completed.")
    print(f"Best object checkpoint: {object_best_checkpoint}")
else:
    object_train_ds = object_val_ds = object_test_ds = None
    object_train_loader = object_val_loader = object_test_loader = None
    object_model = None
    object_trainer = None
    object_best_checkpoint = None
    print("Bá» qua object pipeline theo cáº¥u hÃ¬nh TRAINING_MODE.")

## 6. Pipeline 2: Attribute Classification

In [ ]:
if RUN_ATTRIBUTE_PIPELINE:
    print("========== PIPELINE 2: ATTRIBUTE CLASSIFICATION ==========")
    from src.training.attribute_trainer import AttributeTrainer
    from src.models.task1.attribute_classifier import AttributeClassifier
    from src.data.task1_dataset import build_task1_datasets
    from torch.utils.data import DataLoader

    if not TASK1_LEARNABLE_BACKBONE and not TASK1_ATTRIBUTE_USE_CACHE:
        raise ValueError("Task 1 attribute pipeline requires feature cache when learnable_backbone=false.")

    print("Khá»Ÿi táº¡o Dataset cho attribute pipeline...")
    attribute_train_ds, attribute_val_ds, attribute_test_ds = build_task1_datasets(
        processed_dir=str(TASK1_PROCESSED_DIR),
        image_dir=str(image_dir),
        roi_size=TASK1_ROI_SIZE,
        use_feature_cache=TASK1_ATTRIBUTE_USE_CACHE,
        feature_cache_dir=str(TASK1_FEATURE_DIR),
        input_mode=TASK1_ATTRIBUTE_INPUT_MODE,
        max_samples=MAX_SAMPLES,
        train_horizontal_flip_p=float(TASK1_CONFIG.augmentation.random_horizontal_flip),
        train_color_jitter=bool(TASK1_CONFIG.augmentation.color_jitter.enabled),
        train_brightness=float(TASK1_CONFIG.augmentation.color_jitter.brightness),
        train_contrast=float(TASK1_CONFIG.augmentation.color_jitter.contrast),
        train_saturation=float(TASK1_CONFIG.augmentation.color_jitter.saturation),
        train_hue=float(TASK1_CONFIG.augmentation.color_jitter.hue),
        train_random_erasing_p=float(TASK1_CONFIG.augmentation.random_erasing_p),
        train_resize_delta=int(TASK1_CONFIG.augmentation.resize_delta),
        mean=FEATURE_MEAN,
        std=FEATURE_STD,
    )

    attribute_model = AttributeClassifier(
        num_attributes=attribute_train_ds.num_attributes,
        feature_dim=int(TASK1_CONFIG.backbone.feature_dim),
        hidden_dim=TASK1_ATTRIBUTE_HIDDEN_DIM,
        dropout=TASK1_ATTRIBUTE_DROPOUT,
        num_layers=TASK1_ATTRIBUTE_NUM_LAYERS,
        backbone_name=str(TASK1_CONFIG.backbone.name) if TASK1_LEARNABLE_BACKBONE else None,
        pretrained=bool(TASK1_CONFIG.backbone.pretrained),
        freeze_backbone=bool(TASK1_CONFIG.backbone.freeze_backbone),
        learnable_backbone=TASK1_LEARNABLE_BACKBONE,
        device=DEVICE,
    )

    attribute_optimizer = torch.optim.AdamW(
        attribute_model.parameters(),
        lr=TASK1_LR,
        weight_decay=float(BASE_CONFIG.optimizer.weight_decay),
    )

    attribute_train_loader = DataLoader(
        attribute_train_ds,
        batch_size=TASK1_BATCH_SIZE,
        shuffle=True,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    attribute_val_loader = DataLoader(
        attribute_val_ds,
        batch_size=TASK1_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    attribute_test_loader = DataLoader(
        attribute_test_ds,
        batch_size=TASK1_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )

    attribute_trainer = AttributeTrainer(
        model=attribute_model,
        train_loader=attribute_train_loader,
        val_loader=attribute_val_loader,
        optimizer=attribute_optimizer,
        use_auto_class_weights=True,
        attribute_threshold_mode=str(TASK1_CONFIG.eval.attribute_threshold_mode),
        attribute_threshold=float(TASK1_CONFIG.eval.attribute_threshold),
        attribute_threshold_scale=float(TASK1_CONFIG.eval.attribute_threshold_scale),
        attribute_threshold_min=float(TASK1_CONFIG.eval.attribute_threshold_min),
        attribute_threshold_max=float(TASK1_CONFIG.eval.attribute_threshold_max),
        freeze_backbone=bool(TASK1_CONFIG.backbone.freeze_backbone),
        freeze_epochs=int(TASK1_CONFIG.backbone.freeze_epochs),
        max_epochs=TASK1_MAX_EPOCHS,
        early_stopping_patience=int(BASE_CONFIG.training.early_stopping_patience),
        gradient_clip_val=float(BASE_CONFIG.training.gradient_clip_val),
        log_every_n_steps=int(BASE_CONFIG.training.log_every_n_steps),
        use_amp=(DEVICE == 'cuda'),
    )

    print("Báº¯t Ä‘áº§u training attribute pipeline...")
    attribute_val_metrics = attribute_trainer.train()
    attribute_best_checkpoint = attribute_trainer.checkpoint_manager.get_best_checkpoint_path()
    print("Attribute pipeline completed.")
    print(f"Best attribute checkpoint: {attribute_best_checkpoint}")
else:
    attribute_train_ds = attribute_val_ds = attribute_test_ds = None
    attribute_train_loader = attribute_val_loader = attribute_test_loader = None
    attribute_model = None
    attribute_trainer = None
    attribute_best_checkpoint = None
    print("Bá» qua attribute pipeline theo cáº¥u hÃ¬nh TRAINING_MODE.")

## 7. Pipeline 3: Relation Classification

In [ ]:
if RUN_RELATION_PIPELINE:
    print("========== PIPELINE 3: RELATION CLASSIFICATION ==========")
    from src.training.relation_trainer import RelationTrainer
    from src.models.task2.relation_classifier import RelationClassifier
    from src.data.task2_dataset import build_task2_datasets
    from torch.utils.data import DataLoader

    if not TASK2_LEARNABLE_BACKBONE and not TASK2_USE_CACHE:
        raise ValueError("Task 2 relation pipeline requires feature cache when learnable_backbone=false.")

    print("Khá»Ÿi táº¡o Dataset cho relation pipeline...")
    relation_train_ds, relation_val_ds, relation_test_ds = build_task2_datasets(
        processed_dir=str(TASK2_PROCESSED_DIR),
        image_dir=str(image_dir),
        roi_size=TASK2_ROI_SIZE,
        use_feature_cache=TASK2_USE_CACHE,
        use_spatial_features=bool(TASK2_CONFIG.spatial.use_spatial_features),
        feature_cache_dir=str(TASK2_FEATURE_DIR),
        input_mode=TASK2_INPUT_MODE,
        max_samples=MAX_SAMPLES,
        train_horizontal_flip_p=float(TASK2_CONFIG.augmentation.random_horizontal_flip),
        train_color_jitter=bool(TASK2_CONFIG.augmentation.color_jitter.enabled),
        train_brightness=float(TASK2_CONFIG.augmentation.color_jitter.brightness),
        train_contrast=float(TASK2_CONFIG.augmentation.color_jitter.contrast),
        train_saturation=float(TASK2_CONFIG.augmentation.color_jitter.saturation),
        train_hue=float(TASK2_CONFIG.augmentation.color_jitter.hue),
        train_random_erasing_p=float(TASK2_CONFIG.augmentation.random_erasing_p),
        train_resize_delta=int(TASK2_CONFIG.augmentation.resize_delta),
        mean=FEATURE_MEAN,
        std=FEATURE_STD,
    )

    relation_model = RelationClassifier(
        num_relations=relation_train_ds.num_relations,
        feature_dim=int(TASK2_CONFIG.backbone.feature_dim),
        spatial_dim=int(TASK2_CONFIG.spatial.spatial_dim),
        hidden_dim=TASK2_HIDDEN_DIM,
        dropout=TASK2_DROPOUT,
        num_layers=TASK2_NUM_LAYERS,
        attention_heads=TASK2_ATTENTION_HEADS,
        fusion_method=TASK2_FUSION_METHOD,
        backbone_name=str(TASK2_CONFIG.backbone.name) if TASK2_LEARNABLE_BACKBONE else None,
        pretrained=bool(TASK2_CONFIG.backbone.pretrained),
        freeze_backbone=bool(TASK2_CONFIG.backbone.freeze_backbone),
        learnable_backbone=TASK2_LEARNABLE_BACKBONE,
        device=DEVICE,
    )

    relation_optimizer = torch.optim.AdamW(
        relation_model.parameters(),
        lr=TASK2_LR,
        weight_decay=float(BASE_CONFIG.optimizer.weight_decay),
    )

    relation_train_loader = DataLoader(
        relation_train_ds,
        batch_size=TASK2_BATCH_SIZE,
        shuffle=True,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    relation_val_loader = DataLoader(
        relation_val_ds,
        batch_size=TASK2_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )
    relation_test_loader = DataLoader(
        relation_test_ds,
        batch_size=TASK2_BATCH_SIZE,
        shuffle=False,
        num_workers=int(BASE_CONFIG.training.num_workers),
        pin_memory=bool(BASE_CONFIG.training.pin_memory and DEVICE == 'cuda'),
    )

    relation_trainer = RelationTrainer(
        model=relation_model,
        train_loader=relation_train_loader,
        val_loader=relation_val_loader,
        optimizer=relation_optimizer,
        label_smoothing=float(TASK2_CONFIG.loss.label_smoothing),
        use_auto_class_weights=True,
        freeze_backbone=bool(TASK2_CONFIG.backbone.freeze_backbone),
        freeze_epochs=int(TASK2_CONFIG.backbone.freeze_epochs),
        max_epochs=TASK2_MAX_EPOCHS,
        early_stopping_patience=int(BASE_CONFIG.training.early_stopping_patience),
        gradient_clip_val=float(BASE_CONFIG.training.gradient_clip_val),
        log_every_n_steps=int(BASE_CONFIG.training.log_every_n_steps),
        use_amp=(DEVICE == 'cuda'),
    )

    print("Báº¯t Ä‘áº§u training relation pipeline...")
    relation_val_metrics = relation_trainer.train()
    relation_best_checkpoint = relation_trainer.checkpoint_manager.get_best_checkpoint_path()
    print("Relation pipeline completed.")
    print(f"Best relation checkpoint: {relation_best_checkpoint}")
else:
    relation_train_ds = relation_val_ds = relation_test_ds = None
    relation_train_loader = relation_val_loader = relation_test_loader = None
    relation_model = None
    relation_trainer = None
    relation_best_checkpoint = None
    print("Bá» qua relation pipeline theo cáº¥u hÃ¬nh TRAINING_MODE.")

## 8. E2E Load & Demo

In [ ]:
from src.evaluation.metrics import compute_classification_metrics, compute_multilabel_metrics
from src.data.dataset import load_vocab
from src.models.e2e import VisualGenomeE2EModel
from src.models.caption.caption_generator import CaptionGenerator


def _resolve_input_tensor(batch, preferred_keys):
    for key in preferred_keys:
        value = batch.get(key)
        if value is not None:
            return value
    raise KeyError(f"KhÃ´ng tÃ¬m tháº¥y input tensor trong batch: {preferred_keys}")


def evaluate_object_model(model, loader):
    model.eval()
    logits_list = []
    targets_list = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            features = _resolve_input_tensor(batch, ['feature', 'object_feature', 'image', 'object_image'])
            logits = model(features)
            logits_list.append(logits.cpu())
            targets_list.append(batch['object_label'].cpu())

    return compute_classification_metrics(torch.cat(logits_list), torch.cat(targets_list))


def evaluate_attribute_model(model, loader):
    model.eval()
    logits_list = []
    targets_list = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            features = _resolve_input_tensor(batch, ['feature', 'attribute_feature', 'image', 'attribute_image'])
            logits = model(features)
            logits_list.append(logits.cpu())
            targets_list.append(batch['attribute_labels'].cpu())

    return compute_multilabel_metrics(
        torch.cat(logits_list),
        torch.cat(targets_list),
        threshold=float(TASK1_CONFIG.eval.attribute_threshold),
        threshold_mode=str(TASK1_CONFIG.eval.attribute_threshold_mode),
        threshold_scale=float(TASK1_CONFIG.eval.attribute_threshold_scale),
        threshold_min=float(TASK1_CONFIG.eval.attribute_threshold_min),
        threshold_max=float(TASK1_CONFIG.eval.attribute_threshold_max),
    )


def evaluate_relation_model(model, loader):
    model.eval()
    logits_list = []
    targets_list = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            features = _resolve_input_tensor(batch, ['feature', 'image', 'union_image'])
            logits = model(features, batch['spatial'])
            logits_list.append(logits.cpu())
            targets_list.append(batch['relation_label'].cpu())

    return compute_classification_metrics(torch.cat(logits_list), torch.cat(targets_list))


if RUN_E2E_WRAPPER:
    if object_best_checkpoint is None or attribute_best_checkpoint is None or relation_best_checkpoint is None:
        raise RuntimeError("E2E wrapper requires all three checkpoints. HÃ£y cháº¡y object, attribute, relation pipelines trÆ°á»›c.")

    e2e_model = VisualGenomeE2EModel.from_models(
        object_model=object_model,
        attribute_model=attribute_model,
        relation_model=relation_model,
    )

    loaded_checkpoints = e2e_model.load_checkpoints(
        object_checkpoint=str(object_best_checkpoint),
        attribute_checkpoint=str(attribute_best_checkpoint),
        relation_checkpoint=str(relation_best_checkpoint),
        strict=True,
    )

    print("E2E wrapper summary:")
    print(e2e_model.save_summary())

    object_test_metrics = evaluate_object_model(e2e_model.object_model, object_test_loader)
    attribute_test_metrics = evaluate_attribute_model(e2e_model.attribute_model, attribute_test_loader)
    relation_test_metrics = evaluate_relation_model(e2e_model.relation_model, relation_test_loader)

    print('Object test metrics:')
    print(object_test_metrics)
    print('Attribute test metrics:')
    print(attribute_test_metrics)
    print('Relation test metrics:')
    print(relation_test_metrics)

    object_vocab = load_vocab(str(TASK1_PROCESSED_DIR / 'object_vocab.json'))
    attribute_vocab = load_vocab(str(TASK1_PROCESSED_DIR / 'attribute_vocab.json'))
    relation_vocab = load_vocab(str(TASK2_PROCESSED_DIR / 'relation_vocab.json'))

    captioner = CaptionGenerator(
        object_vocab=object_vocab,
        attribute_vocab=attribute_vocab,
        relation_vocab=relation_vocab,
    )

    demo_caption = captioner.generate_caption(
        subject_name='person',
        object_name='table',
        attributes=['small', 'wooden'],
        relation='standing near',
        template_idx=0,
    )
    print('Caption demo:')
    print(demo_caption)
else:
    e2e_model = None
    loaded_checkpoints = None
    object_test_metrics = None
    attribute_test_metrics = None
    relation_test_metrics = None
    print('Bá» qua E2E wrapper theo cáº¥u hÃ¬nh TRAINING_MODE.')

## 9. Cleanup CUDA Memory

In [ ]:
from src.utils.memory import cleanup_cuda_memory

cleanup_cuda_memory(note='HoÃ n táº¥t pipeline')